# Data collection and cleaning
Collects data from Arxiv, does basic data cleaning, and uploads it to supabase.

Requirements:
requests
feedparser
tqdm
supabase

In [12]:
import requests
import feedparser
from datetime import datetime, timedelta
import tarfile, io, os, re, time
import shutil
import tempfile
from tqdm import tqdm
from supabase import create_client, Client
import pandas as pd
from pprint import pp


In [ ]:
# base url for the arxiv api. 
BASE_URL = "http://export.arxiv.org/api/query"

# Do not change these parameters. Arxiv specifically asks for a 3 second delay between API calls
ARXIV_MAX_PER_REQUEST = 1000
SLEEP_TIME = 3

# These parameters can be changed as needed.
CATEGORIES = ["cs.DS", "cs.IT", "cs.CC", "math.CO"]
DAYS_BACK = 7
KEYWORDS = [
      "locally+decodable+code",
      "matrix+concentration",
      "coding+theory",
      "hypergraph",
      "random+tensor",
      "matching+vectors",
      "rainbow+cycle",
    ]

def fetch_arxiv_metadata(categories, max_results, days_back=DAYS_BACK, keywords=None):
    """fetches metadata of papers uploaded to Arxiv in the specified `categories` up to 
    `days_back` days ago, which contain any specified `keywords`. 
    """
    cutoff_date = datetime.now() - timedelta(days=days_back)
    collected_papers = []
    skipped_papers = []
    failed_papers = []

    query = '(' + " OR ".join(f"cat:{c}" for c in categories) + ')'
    if keywords:
        keywords = [f"abs:\"{kw}\"" for kw in keywords]
        query += f" AND (" + " OR ".join(keywords) + ')'

    # Paginated fetch from the arxiv api. Will never need more than
    # one iteration for nightly pipeline, but this allows the same code
    # to be reused for the intial fetch.
    start = 0
    collected = 0
    while collected < max_results:
        params = {
            "search_query": query,
            "start": start,
            "max_results": min(ARXIV_MAX_PER_REQUEST, 2 * (max_results - collected)),
            "sortBy": "submittedDate",
            "sortOrder": "descending",
        }
        response = requests.get(BASE_URL, params=params)
        feed = feedparser.parse(response.text)

        if not feed.entries:
            return collected_papers, skipped_papers, failed_papers

        for entry in feed.entries:
            try:
                if (collected == 10):
                    collected += 1;
                    raise Exception("Test exception")
                published = datetime.strptime(entry.updated, "%Y-%m-%dT%H:%M:%SZ")
                if published < cutoff_date:
                    return collected_papers, skipped_papers, failed_papers
                
                paper = {
                    "title": entry.title.strip(),
                    "authors": [a.name for a in entry.authors],
                    "published": published.isoformat(),
                    "abstract": entry.summary.strip(),
                    "id": entry.id.split("/abs/")[-1],
                    "url": entry.link,
                    "arxiv_category": entry.arxiv_primary_category['term']
                }
            except Exception as e:
                log_failure(entry, e, stage="metadata_extraction", output=failed_papers)
                continue

            # Ocassionally, a paper will be marked in many categories, but the primary
            # category will not be relevant (i.e. 'https://arxiv.org/abs/2607.06524v1')
            # we don't want to collect those papers, since they are unlikely to be relevant.
            if entry.arxiv_primary_category['term'] not in categories:
                skipped_papers.append(paper)
                continue

            collected += 1
            collected_papers.append(paper)
        start += ARXIV_MAX_PER_REQUEST
        time.sleep(SLEEP_TIME)
    return collected_papers, skipped_papers, failed_papers

def log_failure(entry, e, stage, output=None):
    """Logs when object `entry` causes an exception `e` at stage `stage` of the processing pipeline.
    Stores results in `output`, or prints to standard output if `output` is None.
    """
    entry_id = getattr(entry, "id", None)

    arxiv_id = None
    if isinstance(entry_id, str) and "/abs/" in entry_id:
        arxiv_id = entry_id.split("/abs/")[-1]

    failure = {
        "stage": stage,
        "failed_at": datetime.now().isoformat(),
        "error_type": type(e).__name__,
        "error_message": str(e),
        "title": getattr(entry, "title", None),
        "arxiv_id": arxiv_id,
        "entry_id": entry_id,
        "url": getattr(entry, "link", None),
    }

    if output is None:
        pp(failure)
    else:
        output.append(failure)

if __name__ == "__main__":
    MAX_RESULTS = 100
    papers, skipped, failed = fetch_arxiv_metadata(CATEGORIES, MAX_RESULTS, DAYS_BACK, keywords=None)
    print(f"Collected {len(papers)} papers")

    # Example: print first paper
    pp(papers[10])
    pp(skipped[3])


Collected 141 papers
{'title': 'Some new results on Sylvester colorings of cubic graphs',
 'authors': ['Luca Ferrarini', 'Vahan Mkrtchyan'],
 'published': '2026-07-07T15:28:55',
 'abstract': 'If $G$ and $H$ are two cubic multi-graphs, then an $H$-coloring '
             'of $G$ is a mapping $f: E(G)\\rightarrow E(H)$, such that for '
             'every $v\\in V(G)$ there is a vertex $x\\in V(H)$, such that '
             '$f(\\partial_G(v))=\\partial_H(x)$. If $G$ admits an '
             '$H$-coloring then it is common to write $H\\prec G$. The '
             'Petersen coloring conjecture predicts that for any bridgeless '
             'cubic graph $G$ one has $P_{10}\\prec G$. Here $P_{10}$ is the '
             'Petersen graph. Let $f: E(G)\\rightarrow E(H)$ be any mapping. '
             'Define: $V(f)=\\{v\\in V(G):\\exists x\\in V(H), '
             'f(\\partial_G(v))=\\partial_H(x)\\}$. Let $S_{10}$ be the '
             'smallest cubic multi-graph that has no perfect matching.

In [3]:
def get_intro_text(session, arxiv_id):
    # --- download source ---
    url = f"https://arxiv.org/e-print/{arxiv_id}"

    try:
        r = session.get(url, timeout=60)
        if r.status_code != 200:
            return ""

        with tarfile.open(fileobj=io.BytesIO(r.content), mode="r:gz") as tar:
            for member in tar:
                if not member.isfile() or not member.name.endswith(".tex"):
                    continue

                f = tar.extractfile(member)
                if not f:
                    continue

                text = f.read().decode(errors="ignore")

                # Check for main document
                if "\\begin{document}" not in text:
                    continue

                # --- extract introduction ---
                m = re.search(
                    r"\\section\*?\{[^}]*[intro|Intro][^}]*\}(.*?)(?=\\section|\Z)",
                    text,
                    re.IGNORECASE | re.DOTALL
                )
                if not m:
                    return ""

                intro = m.group(1)

                # --- minimal LaTeX cleanup ---
                # Remove comments
                intro = re.sub(r"%.*", "", intro)

                # # Remove common commands (light cleanup)
                # intro = re.sub(r"\\[a-zA-Z]+\{.*?\}", "", intro)
                # intro = re.sub(r"\\[a-zA-Z]+", "", intro)
                # # 2. Replace all block math environments with a placeholder
                # intro = re.sub(r"\\[.*?\\]", " <MATH> ", intro, flags=re.DOTALL) # Display math (\[...\])
                # intro = re.sub(
                #     r"\\begin\{equation\}.*?\\end\{equation\}",
                #     " <MATH> ", intro, flags=re.DOTALL
                # )

                # # 3. Replace all inline math ($...$) with a placeholder
                # intro = re.sub(r"\$.*?\$", " <MATH> ", intro, flags=re.DOTALL)

                # # 4. Preserve content of commands with arguments (e.g., \textbf{content} -> content)
                # intro = re.sub(r"\\[a-zA-Z]+\*?\{([^}]*)\}", r"\1", intro)

                # # 5. Remove any remaining backslash-prefixed commands or artifacts (like \1, \item, \section etc.)
                # intro = re.sub(r"\\[^\\s]+", "", intro)

                # 6. Normalize whitespace
                intro = re.sub(r"\\s+", " ", intro)
                return intro.strip()
    except (tarfile.ReadError, OSError):
        return ""
    return ""